In [ ]:
import sys
sys.path.insert(0, '/content/Landmark-aligmenttest/CVLface-main/cvlface')


# TinyFace alignment benchmark
Chạy DFA MobileNet, DFA ResNet50, SCRFD-10G-KPS và MediaPipe trên toàn bộ TinyFace. Dataset/model/checkpoint không được commit vào GitHub.

In [ ]:
# Cell 1 — clone, GPU, Drive và cấu trúc thư mục
from pathlib import Path
import os, subprocess, sys
ROOT = Path('/content/Landmark-aligmenttest')
if not ROOT.exists():
    subprocess.check_call(['git','clone','https://github.com/Tuancoolboy/Landmark-aligmenttest.git',str(ROOT)])
os.chdir(ROOT)
subprocess.run(['nvidia-smi'], check=False)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_RESULTS = Path('/content/drive/MyDrive/tinyface_alignment_results')
DATA_ROOT = ROOT/'data'/'tinyface'
RESULT_ROOT = DRIVE_RESULTS
DATA_ROOT.mkdir(parents=True, exist_ok=True); RESULT_ROOT.mkdir(parents=True, exist_ok=True)
print(ROOT, DATA_ROOT, RESULT_ROOT)

In [ ]:
# Cell 2 — HF token từ Colab Secrets (không hard-code)
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
if not HF_TOKEN: raise RuntimeError('Tạo Colab Secret tên HF_TOKEN trước khi chạy')
os.environ['HF_TOKEN'] = HF_TOKEN
print('HF token loaded from Colab Secrets')

In [ ]:
# Cell 3 — tải ZIP TinyFace và chuẩn hóa /data/tinyface
ZIP_URL = 'https://drive.google.com/uc?id=1xTZc7lNmWN33ECO2AKH6FycGdiqIK7W0'
ZIP_PATH = ROOT/'tinyface.zip'
if not (DATA_ROOT/'Testing_Set'/'Probe').is_dir():
    subprocess.check_call([sys.executable,'-m','pip','install','-q','gdown'])
    subprocess.check_call(['gdown','--fuzzy',ZIP_URL,'-O',str(ZIP_PATH)])
    import zipfile, shutil, tempfile
    with tempfile.TemporaryDirectory() as td:
        with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(td)
        candidates = [Path(td)] + [p for p in Path(td).rglob('*') if p.is_dir()]
        src = next((p for p in candidates if (p/'Testing_Set'/'Probe').is_dir()), None)
        if src is None: raise RuntimeError('ZIP không chứa Testing_Set/Probe')
        if DATA_ROOT.exists(): shutil.rmtree(DATA_ROOT)
        shutil.copytree(src, DATA_ROOT)
for name in ['Probe','Gallery_Match','Gallery_Distractor']:
    p=DATA_ROOT/'Testing_Set'/name
    if not p.is_dir(): raise FileNotFoundError(p)
    print(name, len(list(p.glob('*.jpg'))))
print('TinyFace:', DATA_ROOT)

In [ ]:
# Cell 4 — dependencies
pkgs = ['numpy==1.26.4','torchvision','transformers==4.34.1','huggingface-hub','omegaconf','opencv-python-headless','onnxruntime-gpu','insightface==0.7.3','mediapipe==0.10.14','pandas','pillow','tqdm','matplotlib','scikit-image']
subprocess.check_call([sys.executable,'-m','pip','install','-q'] + pkgs)
print('dependencies installed')

In [ ]:
# Cell 5 — benchmark engine
import json, time, cv2, numpy as np, torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from huggingface_hub import snapshot_download
from general_utils.huggingface_model_utils import load_model_from_local_path
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda': raise RuntimeError('Chọn GPU runtime trong Colab')
CACHE = RESULT_ROOT/'model_cache'; CACHE.mkdir(exist_ok=True)
def get_model(repo):
    path = CACHE/repo.replace('/','__')
    marker = path/'config.json'
    if not marker.exists(): snapshot_download(repo_id=repo, local_dir=str(path), token=HF_TOKEN)
    return load_model_from_local_path(str(path)).to(DEVICE).eval()
recognizer = get_model('minchul/cvlface_adaface_vit_base_kprpe_webface4m')
dfa = {'dfa_mobilenet':get_model('minchul/cvlface_DFA_mobilenet'), 'dfa_resnet50':get_model('minchul/cvlface_DFA_resnet50')}
from insightface.app import FaceAnalysis
scrfd = FaceAnalysis(name='buffalo_l', root=str(CACHE/'insightface'), allowed_modules=['detection'], providers=['CUDAExecutionProvider','CPUExecutionProvider'])
scrfd.prepare(ctx_id=0, det_thresh=0.3, det_size=(640,640))
import mediapipe as mp
mesh = mp.solutions.face_mesh.FaceMesh(static_image_mode=True,max_num_faces=1,refine_landmarks=False,min_detection_confidence=0.3)
ARC = np.array([[38.2946,51.6963],[73.5318,51.5014],[56.0252,71.7366],[41.5493,92.3655],[70.7299,92.2041]],np.float32)
def square(path):
    x=np.asarray(Image.open(path).convert('RGB')); h,w=x.shape[:2]; s=max(h,w); y=np.zeros((s,s,3),np.uint8); y[(s-h)//2:(s-h)//2+h,(s-w)//2:(s-w)//2+w]=x; return y
def tensor(x): return torch.from_numpy(x.copy()).permute(2,0,1).float().div(127.5).sub(1).unsqueeze(0).to(DEVICE)
def warp(x,k):
    M,_=cv2.estimateAffinePartial2D(k.astype(np.float32),ARC,method=cv2.LMEDS)
    if M is None: raise RuntimeError('similarity transform failed')
    y=cv2.warpAffine(x,M,(112,112),borderValue=0); ak=np.concatenate([k,np.ones((5,1),np.float32)],1)@M.T
    return tensor(y), torch.from_numpy(ak/112).float()[None].to(DEVICE)
def infer(path,name):
    x=square(path)
    if name.startswith('dfa_'):
        o=dfa[name](tensor(x)); return o[0],o[2],float(o[3].item())
    if name=='scrfd10g':
        fs=scrfd.get(cv2.cvtColor(x,cv2.COLOR_RGB2BGR));
        if not fs: raise RuntimeError('no face')
        f=max(fs,key=lambda z: float((z.bbox[2]-z.bbox[0])*(z.bbox[3]-z.bbox[1]))); a,k=warp(x,np.asarray(f.kps,np.float32)); return a,k,float(f.det_score)
    e=cv2.resize(x,(640,640),interpolation=cv2.INTER_CUBIC); r=mesh.process(e)
    if not r.multi_face_landmarks: raise RuntimeError('no face mesh')
    p=r.multi_face_landmarks[0].landmark
    def mean(ii): return np.mean([[p[i].x*x.shape[1],p[i].y*x.shape[0]] for i in ii],axis=0)
    k=np.asarray([mean([33,133]),mean([362,263]),mean([1]),mean([61]),mean([291])],np.float32); a,k=warp(x,k); return a,k,1.0
def emb(a,k): return F.normalize(recognizer(a,k).float(),dim=1)[0].detach().cpu().numpy().astype('float32')
print('loaded on',DEVICE)

In [ ]:
# Cell 6 — preflight và full benchmark/resume
from collections import defaultdict
sets={'probe':sorted((DATA_ROOT/'Testing_Set/Probe').glob('*.jpg')), 'gallery_match':sorted((DATA_ROOT/'Testing_Set/Gallery_Match').glob('*.jpg')), 'distractor':sorted((DATA_ROOT/'Testing_Set/Gallery_Distractor').glob('*.jpg'))}
PIPELINES=['dfa_mobilenet','dfa_resnet50','scrfd10g','mediapipe']
def label(p): return int(p.stem.split('_')[0])
def size_bin(p):
    with Image.open(p) as im: s=min(im.size)
    return 'S1 (<=16 px)' if s<=16 else 'S2 (17-20 px)' if s<=20 else 'S3 (21-24 px)' if s<=24 else 'S4 (25-28 px)' if s<=28 else 'S5 (29-32 px)' if s<=32 else 'S6 (>32 px)'
all_items=[(p,k) for k,ps in sets.items() for p in ps]
for name in PIPELINES:
    a,k,c=infer(all_items[0][0],name); assert tuple(a.shape[-2:])==(112,112); assert tuple(k.shape[-2:])==(5,2); assert emb(a,k).shape==(512,); print(name,'OK')
def run(name):
    cp=RESULT_ROOT/'checkpoints'/f'{name}.npz'; cp.parent.mkdir(exist_ok=True)
    rows=[]; start=0
    if cp.exists():
        z=np.load(cp,allow_pickle=True); rows=list(z['rows']); start=int(z['next'])
        rows=[r.item() if hasattr(r,'item') else r for r in rows]; print('resume',name,start)
    with torch.inference_mode():
        for i,(path,split) in enumerate(tqdm(all_items[start:],desc=name),start):
            t=time.perf_counter(); row={'path':str(path),'split':split,'label':label(path) if split!='distractor' else -100,'size_bin':size_bin(path),'success':False,'error':'','embedding':None}
            try:
                a,k,c=infer(path,name); row['confidence']=c; row['embedding']=emb(a,k).tolist(); row['success']=c>=0.3
            except Exception as e: row['error']=f'{type(e).__name__}: {e}'
            row['latency_ms']=(time.perf_counter()-t)*1000; rows.append(row)
            if (i+1)%100==0 or i+1==len(all_items): np.savez_compressed(cp,rows=np.array(rows,dtype=object),next=i+1)
    return rows
results={name:run(name) for name in PIPELINES}
print('completed', {k:len(v) for k,v in results.items()})

In [ ]:
# Cell 7 — Rank-1/5/20, coverage, latency, CSV/JSON/Markdown
import pandas as pd, matplotlib.pyplot as plt
def metrics(rows):
    valid=[r for r in rows if r['success'] and r['embedding'] is not None]; by={r['path']:np.asarray(r['embedding'],np.float32) for r in valid}
    gallery=[r for r in rows if r['split']!='probe' and r['path'] in by]; probes=[r for r in rows if r['split']=='probe']
    G=np.asarray([by[r['path']] for r in gallery]); gl=np.asarray([r['label'] for r in gallery]); ranks={1:[],5:[],20:[]}
    for p in probes:
        if p['path'] not in by or len(G)==0:
            for q in ranks:ranks[q].append(0.0)
            continue
        order=np.argsort(-(G@by[p['path']]))
        hit=(gl[order]==p['label'])
        for q in ranks:ranks[q].append(float(hit[:q].any()))
    lat=[r['latency_ms'] for r in rows]; return {'count':len(rows),'coverage':sum(r['success'] for r in rows)/len(rows),'rank1':100*np.mean(ranks[1]),'rank5':100*np.mean(ranks[5]),'rank20':100*np.mean(ranks[20]),'latency_p50_ms':float(np.percentile(lat,50)),'latency_p95_ms':float(np.percentile(lat,95)),'failures':sum(not r['success'] for r in rows)}
summary=pd.DataFrame([{'pipeline':n,**metrics(rs)} for n,rs in results.items()]); summary.to_csv(RESULT_ROOT/'summary.csv',index=False)
payload={n:metrics(rs) for n,rs in results.items()}; (RESULT_ROOT/'results.json').write_text(json.dumps(payload,indent=2))
report=['# TinyFace alignment benchmark','', 'Single-view, no horizontal-flip TTA. TinyFace has no landmark ground truth; metrics are downstream recognition metrics.','',summary.to_markdown(index=False)]
(RESULT_ROOT/'benchmark_report.md').write_text('\n'.join(report)); display(summary)
summary.set_index('pipeline')[['rank1','rank5','rank20']].plot.bar(); plt.ylabel('percent'); plt.tight_layout(); plt.savefig(RESULT_ROOT/'rank_by_pipeline.png',dpi=160); print('saved to',RESULT_ROOT)